In [1]:
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger("util").setLevel(logging.ERROR)

import numpy as np 
import pandas as pd 
from scipy.optimize import minimize
from utils import get_historical_data, lgb_ranker, optimal_weights, preprocessing
from sklearn.model_selection import train_test_split, KFold

#Used to solve the constraint maximization
#%pip install cvxpy/

df = pd.read_csv('fpl_player_statistics.csv')

In [2]:
df_players = get_historical_data('merged_gw_25_26.csv', 6, 12, 'fixtures_25_26.csv')

In [3]:
df_players.columns

Index(['name', 'total_points', 'GW', 'fixture_difficulty', 'was_home', 'value',
       'total_points_so_far', 'form', 'xP', 'xGI', 'next_points',
       'next_fixture_difficulty', 'next_home_ratio'],
      dtype='object')

In [4]:
df_players = df_players[~df_players['form'].isna()]
df_players.index = range(df_players.shape[0])

#'xGI', 'next_home_ratio',
# 'xP',
cols_x = ['total_points_so_far', 'form', 'name', 'next_fixture_difficulty',   'value']

feature_cols = [col for col in cols_x if col not in ['name']]
cols_y = ['next_points']

X = df_players[cols_x]
y = df_players[cols_y]

avg_val_prec_wts = 0
avg_train_prec_wts  = 0

avg_val_ndcg_wts = 0
avg_train_ndcg_wts  = 0

avg_val_prec_lgb = 0
avg_train_prec_lgb  = 0

avg_val_ndcg_lgb = 0
avg_train_ndcg_lgb  = 0

#2-fold split since accuracy of top predictions is dependent on dataset size, so keep size of train and val the same
kf = KFold(n_splits=2, shuffle=True, random_state=42)

k = 0
for train_index, test_index in kf.split(X):

    k+=1
    print (f"Fold number : {k}")

    X_train, X_val = X.iloc[train_index], X.iloc[test_index]
    y_train, y_val = y.iloc[train_index], y.iloc[test_index]

    X_train, X_val = preprocessing(X_train, X_val, feature_cols)
        
    ndcg, ranker = lgb_ranker(X_train, X_val, y_train, y_val, feature_cols)
    val_ndcg = optimal_weights(X_train, X_val, y_train, y_val)

    #avg_val_prec_wts += val_prec/2
    #avg_train_prec_wts += train_prec/2
    #avg_train_ndcg_wts += train_ndcg/2
    #avg_val_prec_lgb += precision/2
    
    avg_val_ndcg_wts += val_ndcg/2
    avg_val_ndcg_lgb += ndcg/2

cols_x.remove('name')

#compare with base metric to see if learning improves accuracy
#print (f"Avg train precision : {avg_train_prec_wts:.2f} and for lgb : {avg_train_prec_lgb:.2f}")
#print (f"Avg val precision : {avg_val_prec_wts:.2f} and for lgb : {avg_val_prec_lgb:.2f}")
#print (f"Avg train NDCG : {avg_train_ndcg_wts:.2f} and for lgb : {avg_train_ndcg_lgb:.2f}")
print (f"Avg val NDCG : {avg_val_ndcg_wts:.2f} and for lgb : {avg_val_ndcg_lgb:.2f}")

#print(f"Avg validation precision for LightGBM: {avg_val_prec_lgb:.2f}")
#print(f"Avg validation NDCG for LightGBM: {avg_val_ndcg_lgb:.2f}")

#Now, find optimal weights to use by using the entire dataset
#bit hacky
# X_train, X_val = preprocessing(df_players[cols_x], df_players[cols_x])
# val_prec, val_prec_base, train_prec, train_prec_base = optimal_weights(X_train, X_val)

# print (f"Train precision for entire dataset : {train_prec:.2f} and for base : {train_prec_base:.2f}")

Fold number : 1
train rank range: 0 59
val rank range: 0 64
train rank labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59]
val rank labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64]
Best w1 : 0.10 and w2 : 0.90
Fold number : 2
train rank range: 0 64
val rank range: 0 59
train rank labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64]
val rank labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59]
Best w1 : 0.1

In [5]:
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression

# Mutual information
mi = mutual_info_regression(X[feature_cols], y, random_state=42)
mi_scores = pd.Series(mi, index=feature_cols).sort_values(ascending=False)

print("\nMutual information:")
print(mi_scores.head(15))


Mutual information:
form                       0.546085
total_points_so_far        0.519521
value                      0.172203
next_fixture_difficulty    0.008088
dtype: float64


In [6]:
# Permutation importance
perm = permutation_importance(
    ranker,
    X[feature_cols],
    y,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

perm_importance = pd.Series(
    perm.importances_mean,
    index=feature_cols
).sort_values(ascending=False)

print("\nPermutation importance:")
print(perm_importance.head(15))


Permutation importance:
total_points_so_far        0.037039
value                      0.015540
next_fixture_difficulty    0.001065
form                       0.000000
dtype: float64


In [7]:
rank_players = pd.read_csv('fpl_player_statistics.csv')

rank_players['total_points_so_far'] = rank_players['total_points']
rank_players['form'] = rank_players['form'].replace(np.nan, 0)
rank_players['value'] = rank_players['now_cost'] * 10

#xP for last 6 games
#rank_players['xP'] = rank_players['points_per_game']

team_mapping = pd.read_csv('fpl_team_mapping.csv')
team_id_to_club = team_mapping.set_index('team_id')['club_name']
fixtures = pd.read_csv('fixtures_26_27.csv')
fixtures['kickoff_time'] = pd.to_datetime(fixtures['kickoff_time'], utc=True)

upcoming = fixtures.loc[~fixtures['finished']].copy()
upcoming['home_club'] = upcoming['team_h'].map(team_id_to_club)
upcoming['away_club'] = upcoming['team_a'].map(team_id_to_club)
if upcoming[['home_club', 'away_club']].isna().any().any():
    raise ValueError('A fixture team ID is missing from fpl_team_mapping.csv')

home_fixtures = upcoming[
    ['id', 'kickoff_time', 'home_club', 'team_h_difficulty']
].rename(columns={'home_club': 'club_name', 'team_h_difficulty': 'difficulty'})
away_fixtures = upcoming[
    ['id', 'kickoff_time', 'away_club', 'team_a_difficulty']
].rename(columns={'away_club': 'club_name', 'team_a_difficulty': 'difficulty'})
team_fixtures = pd.concat([home_fixtures, away_fixtures], ignore_index=True)
team_fixtures = team_fixtures.sort_values(
    ['club_name', 'kickoff_time', 'id'], kind='stable'
)
team_fixtures['fixture_number'] = team_fixtures.groupby('club_name').cumcount()
next_six_fixtures = team_fixtures[team_fixtures['fixture_number'] < 6]
fixture_counts = next_six_fixtures.groupby('club_name').size()
if (fixture_counts < 6).any():
    raise ValueError('At least one club has fewer than six upcoming fixtures')

next_six_difficulty = next_six_fixtures.groupby('club_name')['difficulty'].mean()
rank_players['next_fixture_difficulty'] = rank_players['club_name'].map(
    next_six_difficulty
)
if rank_players['next_fixture_difficulty'].isna().any():
    missing_clubs = rank_players.loc[
        rank_players['next_fixture_difficulty'].isna(), 'club_name'
    ].unique()
    raise ValueError(f'No upcoming fixture difficulty found for clubs: {missing_clubs}')

rank_features = rank_players[feature_cols].apply(pd.to_numeric, errors='coerce')
rank_features = rank_features.fillna(X[feature_cols].median())
rank_players['ranker_score'] = ranker.predict(rank_features)
ranked_players = rank_players.sort_values(
    'ranker_score', ascending=False
).reset_index(drop=True)
ranked_players['predicted_rank'] = ranked_players.index + 1

ranked_players[
    [
        'predicted_rank', 'player_name', 'club_name', 'position_name',
        'next_fixture_difficulty', 'ranker_score'
    ]
].head(40)

,predicted_rank,player_name,club_name,position_name,next_fixture_difficulty,ranker_score
0,1,Bruno B.Fernandes,Man Utd,MID,3.000000,0.581222
1,2,Cody Gakpo,Liverpool,MID,2.833333,0.581222
2,3,Harvey Barnes,Newcastle,MID,2.666667,0.559157
3,4,Rayan Cherki,Man City,MID,2.833333,0.556055
4,5,Cole Palmer,Chelsea,MID,3.000000,0.556055
5,6,Antoine Semenyo,Man City,MID,2.833333,0.556055
6,7,Morgan Rogers,Chelsea,MID,3.000000,0.556055
7,8,Alexander Isak,Liverpool,FWD,2.833333,0.556055
8,9,João Pedro João Pedro,Chelsea,FWD,3.000000,0.556055
9,10,Bryan Mbeumo,Man Utd,MID,3.000000,0.556055


In [ ]:
# means = {}
# stds = {}
# for col in ['form', 'total_points']:
#     means[col] = df_players[col].mean()
#     stds[col] = df_players[col].std()
#     df[col + '_norm'] = (df[col] - means[col])/stds[col]

# df['metric'] = df['form_norm'] * 0.85 + df['total_points_norm'] * 0.15

In [49]:
df.index = range(df.shape[0])

In [50]:
import cvxpy as cp

values = df['metric']
costs = df['now_cost'].tolist()
positions = df['position_name'].tolist()
budget = 84.1                   # Budget constraint - select only 11 players and select 
                                # the 4 substitutes with minimum value players
                                 
n = len(values)            
max_items = 11

# Decision variables
x = cp.Variable(n, boolean=True)  # Binary variables: 1 if player is selected, 0 otherwise

# Objective function
objective = cp.Maximize(cp.sum(cp.multiply(values, x)))

fwd_indices = df[df['position_name']=='FWD'].index.tolist()
mid_indices = df[df['position_name']=='MID'].index.tolist()
def_indices = df[df['position_name']=='DEF'].index.tolist()
gkp_indices = df[df['position_name']=='GKP'].index.tolist()

# Constraint: total cost ≤ budget, number of players in each position, and total number of players selected
constraints = [
    cp.sum(cp.multiply(costs, x)) <= budget,
    cp.sum(x) <= max_items,
    cp.sum(x[fwd_indices]) <= 3,
    cp.sum(x[mid_indices]) <= 4,
    cp.sum(x[def_indices]) <= 4,
    cp.sum(x[gkp_indices]) == 1
    ]

problem = cp.Problem(objective, constraints)
problem.solve()

116.70516647839494

In [55]:
#print the selected players after GW 32 for 2025-26 season
df[['player_name', 'position_name', 'now_cost', 'form', 'total_points', 'metric']][x.value.astype(bool)].sort_values(by='position_name')

,player_name,position_name,now_cost,form,total_points,metric
2,Gabriel Gabriel,DEF,7.1,2.0,177,10.384654
9,Marc Guéhi,DEF,5.1,5.0,150,10.289352
52,Neco N.Williams,DEF,4.8,8.7,115,10.074871
0,Erling Haaland,FWD,14.5,0.7,199,11.013304
4,Igor Thiago Thiago,FWD,7.3,6.3,168,11.941682
8,Jarrod Bowen,FWD,7.6,5.3,157,10.832959
37,Caoimhín Kelleher,GKP,4.8,5.7,122,9.029684
1,Bruno B.Fernandes,MID,10.3,5.7,193,13.078422
3,Antoine Semenyo,MID,8.2,0.7,176,9.701741
5,Declan Rice,MID,7.2,2.0,167,9.814409


In [58]:
#Overall top 10 performing player prediction after GW 32 for 2025-26 season
df[['player_name', 'position_name', 'now_cost', 'form', 'total_points', 'metric']].iloc[:10].sort_values(by='metric', ascending=False)

,player_name,position_name,now_cost,form,total_points,metric
1,Bruno B.Fernandes,MID,10.3,5.7,193,13.078422
4,Igor Thiago Thiago,FWD,7.3,6.3,168,11.941682
0,Erling Haaland,FWD,14.5,0.7,199,11.013304
8,Jarrod Bowen,FWD,7.6,5.3,157,10.832959
7,Harry Wilson,MID,6.0,4.7,157,10.544088
2,Gabriel Gabriel,DEF,7.1,2.0,177,10.384654
9,Marc Guéhi,DEF,5.1,5.0,150,10.289352
5,Declan Rice,MID,7.2,2.0,167,9.814409
3,Antoine Semenyo,MID,8.2,0.7,176,9.701741
6,João Pedro João Pedro,FWD,7.7,1.0,166,9.275932


I use the base metric at the start of the season, and optimized weight metric for wildcard/free hit. I'm trying to see if the metric can be better defined as the score output of a binary classification model, and/or using other variables. In addition, adding data for the past 4-5 seasons should make the metric more stable. Let me know your thoughts in the comments!